In [ ]:
!pip install pygam -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from sklearn.model_selection import cross_val_score  # train_test_split & KFold diganti versi custom (replikasi R)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet, ElasticNetCV
from sklearn.pipeline import Pipeline

from pygam import LinearGAM, s

import itertools, warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from google.colab import files
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_excel(fname) if fname.lower().endswith(('.xlsx', '.xls')) else pd.read_csv(fname)

df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()

In [ ]:
df.info()
# Ukuran pusat & sebaran lengkap
desc = df.describe().T
desc['median']   = df.median()
desc['mode']     = df.mode().iloc[0]
desc['IQR']      = df.quantile(0.75) - df.quantile(0.25)
desc['skewness'] = df.skew()
desc['kurtosis'] = df.kurt()
desc

In [ ]:
print("Missing values per kolom:")
print(df.isnull().sum())
print("\nJumlah baris duplikat:", df.duplicated().sum())

In [ ]:
# Distribusi tiap variabel
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, df.columns):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribusi {col}')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot per variabel (deteksi outlier visual)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, df.columns):
    sns.boxplot(y=df[col], ax=ax, color='lightcoral')
    ax.set_title(col)
plt.tight_layout()
plt.show()

def iqr_outlier_count(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lb) | (s > ub)).sum()

print("Jumlah outlier (aturan IQR 1.5x) per kolom:")
print(df.apply(iqr_outlier_count))

In [ ]:
# Korelasi antar variabel
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriks Korelasi')
plt.show()

In [ ]:
# Hubungan tiap fitur terhadap target (regresi linear sederhana per fitur)
target = 'PE'
features = [c for c in df.columns if c != target]

fig, axes = plt.subplots(1, len(features), figsize=(18, 4))
for ax, col in zip(axes, features):
    sns.regplot(x=df[col], y=df[target], ax=ax,
                scatter_kws={'alpha': 0.15, 's': 10}, line_kws={'color': 'red'})
    ax.set_title(f'{col} vs {target}')
plt.tight_layout()
plt.show()

In [ ]:
# Multikolinearitas (VIF)
X_check = sm.add_constant(df[features])
vif_df = pd.DataFrame({
    'variabel': X_check.columns,
    'VIF': [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])]
})
vif_df

In [ ]:
# Model linear awal (baseline) untuk cek residual
model0 = smf.ols(f'{target} ~ ' + ' + '.join(features), data=df).fit()
resid = model0.resid

# Normalitas residual
sample_resid = resid.sample(min(5000, len(resid)), random_state=RANDOM_STATE)
shapiro_stat, shapiro_p = stats.shapiro(sample_resid)
print(f"Shapiro-Wilk: statistic={shapiro_stat:.4f}, p-value={shapiro_p:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(resid, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Residual')
sm.qqplot(resid, line='45', ax=axes[1])
axes[1].set_title('QQ-Plot Residual')
plt.tight_layout()
plt.show()

# Homoskedastisitas
bp_stat, bp_p, _, _ = het_breuschpagan(resid, model0.model.exog)
print(f"Breusch-Pagan: statistic={bp_stat:.4f}, p-value={bp_p:.4g}")
print("(p-value kecil -> indikasi heteroskedastisitas, perlu dicatat sebagai catatan interpretasi, bukan alasan transformasi otomatis)")

## Model Selection: AICc vs 10-Fold CV

Mulai dari sini, algoritma disamakan persis dengan `functions_regresi_ccpp.R` (train-test split, pembentukan fold, rumus AICc, grid hyperparameter, dan cara agregasi RMSE). **Catatan**: karena generator bilangan acak R dan Python berbeda, urutan hasil `sample()`/`permutation()` dengan `seed` yang sama tidak akan identik bit-per-bit antar bahasa — yang disamakan adalah *prosedur*/algoritmanya (shuffle penuh lalu ambil bagian pertama sebagai test set, fold round-robin setelah shuffle, dst), bukan angka acak itu sendiri.

In [ ]:
X = df[features].copy()
y = df[target].copy()

def make_train_test_split(n, test_size=0.20, seed=42):
    """Replikasi make_train_test_split() di R: shuffle penuh, lalu
    n_test baris pertama jadi test, sisanya jadi train."""
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    n_test = int(np.ceil(n * test_size))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return train_idx, test_idx

def make_folds(n, K=10, seed=42):
    """Replikasi make_folds() di R: shuffle penuh, lalu fold id
    ditempel round-robin (rep(1:K, length.out = n)), BUKAN blok berurutan
    seperti sklearn KFold(shuffle=True)."""
    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(n)
    fold_id = np.resize(np.arange(1, K + 1), n)
    folds = [np.sort(shuffled[fold_id == k]) for k in range(1, K + 1)]
    return folds

n_total = len(X)
train_idx, test_idx = make_train_test_split(n_total, test_size=0.20, seed=RANDOM_STATE)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test  = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test  = y.iloc[test_idx].reset_index(drop=True)
print("Train:", X_train.shape, " Test:", X_test.shape)

# Skema CV yang SAMA dipakai di semua model agar perbandingan adil
# (folds dibentuk dari make_folds() di atas, replikasi persis skema R)
n_train = X_train.shape[0]
fold_indices = make_folds(n_train, K=10, seed=RANDOM_STATE)
kf_splits = []
for val_idx in fold_indices:
    tr_idx = np.setdiff1d(np.arange(n_train), val_idx)
    kf_splits.append((tr_idx, val_idx))

def compute_aic(n, rss, k):
    return n * np.log(rss / n) + 2 * k

def compute_aicc(n, rss, k):
    aic = compute_aic(n, rss, k)
    if n - k - 1 > 0:
        aic += (2 * k * (k + 1)) / (n - k - 1)
    return aic

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)


In [ ]:
poly_results = []
for degree in [1, 2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    Xp_train = poly.fit_transform(X_train)

    lr = LinearRegression().fit(Xp_train, y_train)
    pred_train = lr.predict(Xp_train)
    rss = np.sum((y_train.values - pred_train) ** 2)
    k = Xp_train.shape[1] + 1  # +1 intercept, sama seperti R: k <- ncol(Xp) + 1
    aicc = compute_aicc(n_train, rss, k)  # AICc (bukan AIC statsmodels), sama seperti R

    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr', LinearRegression())
    ])
    # cv=kf_splits -> pakai fold round-robin custom (replikasi make_folds() R), bukan KFold sklearn
    cv_mse = -cross_val_score(pipe, X_train, y_train, cv=kf_splits, scoring='neg_mean_squared_error')

    poly_results.append({
        'degree': degree,
        'k': k,
        'AICc': aicc,
        'CV_RMSE': np.sqrt(cv_mse.mean())  # = sqrt(mean(fold_rmse^2)), sama seperti R
    })

poly_df = pd.DataFrame(poly_results)
poly_df


In [ ]:
best_deg_aic = poly_df.loc[poly_df['AICc'].idxmin(), 'degree']
best_deg_cv  = poly_df.loc[poly_df['CV_RMSE'].idxmin(), 'degree']
print(f"Derajat terpilih via AICc: {best_deg_aic}")
print(f"Derajat terpilih via CV  : {best_deg_cv}")

rho_poly, p_poly = spearmanr(poly_df['AICc'], poly_df['CV_RMSE'])
print(f"Korelasi Spearman ranking AICc vs CV_RMSE: rho={rho_poly:.3f}, p={p_poly:.3f}")


In [ ]:
def fit_scaler(X):
    """Replikasi fit_scaler() di R: pakai sd sampel (ddof=1), scale=1 jika sd 0/NaN."""
    center = X.mean(axis=0)
    scale = X.std(axis=0, ddof=1)
    scale = scale.where(scale.notna() & (scale != 0), 1.0)
    return center, scale

def apply_scaler(X, center, scale):
    return (X - center) / scale

scaler_center, scaler_scale = fit_scaler(X_train)
X_train_scaled = apply_scaler(X_train, scaler_center, scaler_scale).values
X_test_scaled  = apply_scaler(X_test, scaler_center, scaler_scale).values
n_train = X_train_scaled.shape[0]

# Grid sama persis dengan R: lambda_grid = 10^seq(-4, 1, length.out = 20), diurutkan menurun
alphas = np.sort(np.logspace(-4, 1, 20))[::-1]
# alpha_mix_grid R: c(0.1, 0.5, 0.9, 1.0)
l1_ratios = [0.1, 0.5, 0.9, 1.0]

en_results = []
for l1r in l1_ratios:
    for a in alphas:
        m = ElasticNet(alpha=a, l1_ratio=l1r, max_iter=10000, random_state=RANDOM_STATE)
        m.fit(X_train_scaled, y_train)
        rss = np.sum((y_train.values - m.predict(X_train_scaled)) ** 2)
        k = np.sum(np.abs(m.coef_) > 1e-12) + 1  # threshold sama seperti R (abs(beta[-1]) > 1e-12)
        en_results.append({'l1_ratio': l1r, 'alpha': a, 'k': k,
                            'AICc': compute_aicc(n_train, rss, k)})

en_df = pd.DataFrame(en_results)
best_aicc_en = en_df.loc[en_df['AICc'].idxmin()]
print("Elastic Net terbaik via AICc:")
print(best_aicc_en)


In [ ]:
en_cv = ElasticNetCV(l1_ratio=l1_ratios, alphas=alphas, cv=kf_splits,
                     max_iter=10000, random_state=RANDOM_STATE)
en_cv.fit(X_train_scaled, y_train)
print(f"Elastic Net terbaik via CV: alpha={en_cv.alpha_:.5f}, l1_ratio={en_cv.l1_ratio_}")


In [ ]:
# CV_MSE untuk SETIAP kombinasi (alpha, l1_ratio) agar bisa dikorelasikan dengan AICc
cv_mse_en = []
for l1r in l1_ratios:
    for a in alphas:
        m = ElasticNet(alpha=a, l1_ratio=l1r, max_iter=10000, random_state=RANDOM_STATE)
        scores = cross_val_score(m, X_train_scaled, y_train, cv=kf_splits, scoring='neg_mean_squared_error')
        cv_mse_en.append(-scores.mean())

en_df['CV_MSE'] = cv_mse_en
rho_en, p_en = spearmanr(en_df['AICc'], en_df['CV_MSE'])
print(f"Korelasi Spearman AICc vs CV_MSE (Elastic Net, semua kombinasi): rho={rho_en:.3f}, p={p_en:.3f}")


In [ ]:
X_train_arr = X_train.values
y_train_arr = y_train.values

# k_spline = 20 di setiap smooth term, sama seperti k_spline default R (make_gam_formula(..., k_spline = 20))
# Grid di sini hanya untuk mendekati pencarian otomatis mgcv::gam(method = "GCV.Cp") yang bersifat kontinu
lams_gcv = np.logspace(-3, 3, 25)

gam_gcv = LinearGAM(s(0, n_splines=20) + s(1, n_splines=20) + s(2, n_splines=20) + s(3, n_splines=20)).gridsearch(
    X_train_arr, y_train_arr, lam=lams_gcv, progress=False)

edof = gam_gcv.statistics_['edof']
rss_gam = np.sum((y_train_arr - gam_gcv.predict(X_train_arr)) ** 2)
aicc_gam_gcv = compute_aicc(len(y_train_arr), rss_gam, edof)  # AICc, sama seperti R (compute_aicc), bukan AIC biasa

print("Lambda terpilih via GCV :", gam_gcv.lam)
print(f"Effective DoF: {edof:.2f} | AICc (df=edof): {aicc_gam_gcv:.2f}")


In [ ]:
# CV manual per nilai lambda (skema fold custom yang sama, agar sebanding dgn model lain)
# lambda_grid sama persis dengan R: 10^seq(-3, 3, length.out = 10)
lams_cv = np.logspace(-3, 3, 10)

gam_cv_results = []
for lam in lams_cv:
    fold_rmse = []
    for tr_idx, val_idx in kf_splits:
        g = LinearGAM(s(0, n_splines=20) + s(1, n_splines=20) + s(2, n_splines=20) + s(3, n_splines=20),
                      lam=lam).fit(X_train_arr[tr_idx], y_train_arr[tr_idx])
        pred = g.predict(X_train_arr[val_idx])
        fold_rmse.append(rmse(y_train_arr[val_idx], pred))
    # CV_RMSE = rata-rata RMSE per fold, sama seperti R: mean(fold_rmse) (bukan sqrt(mean(mse)))
    gam_cv_results.append({'lam': lam, 'CV_RMSE': np.mean(fold_rmse)})

gam_cv_df = pd.DataFrame(gam_cv_results)
best_lam_cv = gam_cv_df.loc[gam_cv_df['CV_RMSE'].idxmin(), 'lam']
print("Lambda terpilih via CV manual:", best_lam_cv)
print("Lambda terpilih via GCV      :", gam_gcv.lam)


In [ ]:
summary = pd.DataFrame([
    {
        'Model': 'Polynomial (nested)',
        'Pilihan_AIC': f"degree={best_deg_aic}",
        'Pilihan_CV': f"degree={best_deg_cv}",
        'Sama?': best_deg_aic == best_deg_cv
    },
    {
        'Model': 'Elastic Net',
        'Pilihan_AIC': f"alpha={best_aicc_en['alpha']:.4f}, l1_ratio={best_aicc_en['l1_ratio']}",
        'Pilihan_CV': f"alpha={en_cv.alpha_:.4f}, l1_ratio={en_cv.l1_ratio_}",
        'Sama?': np.isclose(best_aicc_en['alpha'], en_cv.alpha_, rtol=0.2) and best_aicc_en['l1_ratio'] == en_cv.l1_ratio_
    },
    {
        'Model': 'GAM',
        'Pilihan_AIC': f"lambda≈{gam_gcv.lam}",
        'Pilihan_CV': f"lambda≈{best_lam_cv}",
        'Sama?': np.isclose(np.ravel(gam_gcv.lam), best_lam_cv, rtol=0.2).all()
    },
])
pd.set_option('display.max_colwidth', 100)
summary


In [ ]:
# --- Polynomial: evaluasi pilihan AICc vs CV pada X_test ---
for label, deg in [('AICc', best_deg_aic), ('CV', best_deg_cv)]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr', LinearRegression())
    ]).fit(X_train, y_train)
    pred_test = pipe.predict(X_test)
    print(f"Polynomial (pilihan {label}, degree={deg}) -> "
          f"Test RMSE = {rmse(y_test.values, pred_test):.4f}, Test R2 = {r2_score(y_test.values, pred_test):.4f}")

print()
# --- Elastic Net: evaluasi pilihan AICc vs CV pada X_test ---
for label, alpha_sel, l1_sel in [('AICc', best_aicc_en['alpha'], best_aicc_en['l1_ratio']),
                                  ('CV', en_cv.alpha_, en_cv.l1_ratio_)]:
    m = ElasticNet(alpha=alpha_sel, l1_ratio=l1_sel, max_iter=10000, random_state=RANDOM_STATE)
    m.fit(X_train_scaled, y_train)
    pred_test = m.predict(X_test_scaled)
    print(f"Elastic Net (pilihan {label}, alpha={alpha_sel:.4f}, l1_ratio={l1_sel}) -> "
          f"Test RMSE = {rmse(y_test.values, pred_test):.4f}, Test R2 = {r2_score(y_test.values, pred_test):.4f}")

print()
# --- GAM: evaluasi pilihan GCV vs CV pada X_test ---
X_test_arr = X_test.values
pred_test_gam_gcv = gam_gcv.predict(X_test_arr)
print(f"GAM (pilihan GCV, lambda={gam_gcv.lam}) -> "
      f"Test RMSE = {rmse(y_test.values, pred_test_gam_gcv):.4f}, Test R2 = {r2_score(y_test.values, pred_test_gam_gcv):.4f}")

gam_cv_final = LinearGAM(s(0, n_splines=20) + s(1, n_splines=20) + s(2, n_splines=20) + s(3, n_splines=20),
                          lam=best_lam_cv).fit(X_train_arr, y_train_arr)
pred_test_gam_cv = gam_cv_final.predict(X_test_arr)
print(f"GAM (pilihan CV, lambda={best_lam_cv}) -> "
      f"Test RMSE = {rmse(y_test.values, pred_test_gam_cv):.4f}, Test R2 = {r2_score(y_test.values, pred_test_gam_cv):.4f}")

# Catatan: R juga menjalankan eksperimen ini pada beberapa sheet sekaligus (run_full_analysis di
# functions_regresi_ccpp.R) lalu membandingkan model terbaik antar sheet dengan ANOVA/Kruskal-Wallis.
# Notebook ini bekerja pada satu file/sheet hasil upload, sehingga bagian perbandingan-antar-sheet
# tersebut tidak direplikasi di sini.
